# MovieLens 1M Exploratory Data Analysis

This notebook explores the MovieLens 1M dataset to understand:
- Rating distribution
- Sparsity of the user-item matrix
- User activity patterns
- Item popularity long-tail distribution

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Resolve data directory - works from any working directory
cwd = Path.cwd()
# If running from notebooks/, go up one level; if from project root, use data/
if cwd.name == "notebooks":
    DATA_DIR = cwd.parent / "data" / "ml-1m"
else:
    DATA_DIR = cwd / "data" / "ml-1m"
print(f"Using data directory: {DATA_DIR}")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

In [ ]:
# Load ratings.dat
ratings_cols = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_csv(
    DATA_DIR / "ratings.dat",
    sep="::",
    names=ratings_cols,
    engine="python",
    encoding="latin-1"
)
print(f"Ratings shape: {ratings.shape}")
print(f"Unique users: {ratings['user_id'].nunique()}")
print(f"Unique movies: {ratings['movie_id'].nunique()}")
print(f"Rating range: {ratings['rating'].min()} - {ratings['rating'].max()}")
print(f"Timestamp range: {ratings['timestamp'].min()} - {ratings['timestamp'].max()}")
ratings.head()

In [ ]:
# Load movies.dat
movies_cols = ["movie_id", "title", "genres"]
movies = pd.read_csv(
    DATA_DIR / "movies.dat",
    sep="::",
    names=movies_cols,
    engine="python",
    encoding="latin-1"
)
print(f"Movies shape: {movies.shape}")
movies.head()

In [ ]:
# Load users.dat
users_cols = ["user_id", "gender", "age", "occupation", "zip_code"]
users = pd.read_csv(
    DATA_DIR / "users.dat",
    sep="::",
    names=users_cols,
    engine="python",
    encoding="latin-1"
)
print(f"Users shape: {users.shape}")
users.head()

## 1. Rating Distribution

In [ ]:
# Rating distribution
rating_counts = ratings["rating"].value_counts().sort_index()
rating_pct = ratings["rating"].value_counts(normalize=True).sort_index() * 100

print("Rating Distribution:")
for r in sorted(ratings["rating"].unique()):
    print(f"  Rating {r}: {rating_counts[r]:,} ({rating_pct[r]:.1f}%)")

print(f"\nMean rating: {ratings['rating'].mean():.3f}")
print(f"Median rating: {ratings['rating'].median():.3f}")
print(f"Std rating: {ratings['rating'].std():.3f}")

In [ ]:
# Rating histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
axes[0].bar(rating_counts.index.astype(str), rating_counts.values, color="steelblue", edgecolor="black")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Rating Distribution (Count)")
for i, v in enumerate(rating_counts.values):
    axes[0].text(i, v + 10000, f"{v:,}", ha="center", va="bottom")

# Percentage plot
axes[1].bar(rating_pct.index.astype(str), rating_pct.values, color="coral", edgecolor="black")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Percentage")
axes[1].set_title("Rating Distribution (Percentage)")
for i, v in enumerate(rating_pct.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 2. Sparsity Analysis

In [ ]:
# Sparsity calculation
n_users = ratings["user_id"].nunique()
n_items = ratings["movie_id"].nunique()
n_ratings = len(ratings)

total_possible = n_users * n_items
sparsity = 1 - (n_ratings / total_possible)
density = n_ratings / total_possible

print(f"Number of users: {n_users:,}")
print(f"Number of items: {n_items:,}")
print(f"Number of ratings: {n_ratings:,}")
print(f"Total possible user-item pairs: {total_possible:,}")
print(f"Density: {density:.6f} ({density*100:.4f}%)")
print(f"Sparsity: {sparsity:.6f} ({sparsity*100:.4f}%)")

## 3. User Activity Analysis

In [ ]:
# Ratings per user
user_activity = ratings.groupby("user_id").size()

print("User Activity Statistics:")
print(f"  Mean: {user_activity.mean():.1f}")
print(f"  Median: {user_activity.median():.1f}")
print(f"  Std: {user_activity.std():.1f}")
print(f"  Min: {user_activity.min()}")
print(f"  Max: {user_activity.max()}")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th percentile: {user_activity.quantile(p/100):.1f}")

In [ ]:
# User activity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(user_activity, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Ratings per User")
axes[0].set_ylabel("Number of Users")
axes[0].set_title("User Activity Distribution (Linear)")
axes[0].axvline(user_activity.median(), color="red", linestyle="--", label=f"Median: {user_activity.median():.0f}")
axes[0].legend()

# Log-log plot
activity_counts = user_activity.value_counts().sort_index()
axes[1].loglog(activity_counts.index, activity_counts.values, "o", alpha=0.6, markersize=4)
axes[1].set_xlabel("Ratings per User (log)")
axes[1].set_ylabel("Number of Users (log)")
axes[1].set_title("User Activity Distribution (Log-Log)")

plt.tight_layout()
plt.show()

## 4. Item Popularity Long-Tail Analysis

In [ ]:
# Ratings per item (popularity)
item_popularity = ratings.groupby("movie_id").size()

print("Item Popularity Statistics:")
print(f"  Mean: {item_popularity.mean():.1f}")
print(f"  Median: {item_popularity.median():.1f}")
print(f"  Std: {item_popularity.std():.1f}")
print(f"  Min: {item_popularity.min()}")
print(f"  Max: {item_popularity.max()}")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th percentile: {item_popularity.quantile(p/100):.1f}")

In [ ]:
# Item popularity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram (linear)
axes[0].hist(item_popularity, bins=50, color="coral", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Ratings per Movie")
axes[0].set_ylabel("Number of Movies")
axes[0].set_title("Item Popularity Distribution (Linear)")
axes[0].axvline(item_popularity.median(), color="red", linestyle="--", label=f"Median: {item_popularity.median():.0f}")
axes[0].legend()

# Log-log plot (Pareto/Zipf)
from scipy import stats

pop_counts = item_popularity.value_counts().sort_index()
axes[1].loglog(pop_counts.index, pop_counts.values, "o", alpha=0.6, markersize=4, label="Data")

# Fit power law for reference
x = pop_counts.index.values
y = pop_counts.values
mask = (x > 0) & (y > 0)
if mask.sum() > 2:
    slope, intercept, r_value, p_value, std_err = stats.linregress(np.log(x[mask]), np.log(y[mask]))
    x_fit = np.logspace(np.log10(x[mask].min()), np.log10(x[mask].max()), 100)
    y_fit = np.exp(intercept) * x_fit ** slope
    axes[1].loglog(x_fit, y_fit, "r--", label=f"Power law fit: α={-slope:.2f}, R²={r_value**2:.3f}")
    axes[1].legend()

axes[1].set_xlabel("Ratings per Movie (log)")
axes[1].set_ylabel("Number of Movies (log)")
axes[1].set_title("Item Popularity Long Tail (Log-Log)")

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most popular movies
top_movies = item_popularity.nlargest(20)
top_movies_df = movies[movies["movie_id"].isin(top_movies.index)].set_index("movie_id").loc[top_movies.index]
top_movies_df["rating_count"] = top_movies.values
print("Top 20 Most Popular Movies:")
print(top_movies_df[["title", "rating_count", "genres"]].to_string())

In [ ]:
# Bottom 20 least popular movies (with at least 1 rating)
bottom_movies = item_popularity.nsmallest(20)
bottom_movies_df = movies[movies["movie_id"].isin(bottom_movies.index)].set_index("movie_id").loc[bottom_movies.index]
bottom_movies_df["rating_count"] = bottom_movies.values
print("Bottom 20 Least Popular Movies:")
print(bottom_movies_df[["title", "rating_count", "genres"]].to_string())

## 5. Summary Statistics

In [ ]:
print("=" * 50)
print("MOVIELENS 1M - SUMMARY")
print("=" * 50)
print(f"Total ratings: {n_ratings:,}")
print(f"Unique users: {n_users:,}")
print(f"Unique movies: {n_items:,}")
print(f"Sparsity: {sparsity*100:.2f}%")
print(f"Mean rating: {ratings['rating'].mean():.3f}")
print(f"Mean ratings/user: {user_activity.mean():.1f}")
print(f"Median ratings/user: {user_activity.median():.1f}")
print(f"Mean ratings/movie: {item_popularity.mean():.1f}")
print(f"Median ratings/movie: {item_popularity.median():.1f}")
print(f"Power law exponent (α): {-slope:.2f}" if 'slope' in locals() else "Power law exponent: N/A")
print("=" * 50)